# Mini Project 1 - Seattle Coffee Analysis


## Section 1: Overview

- **Name:** Abdulrahman
- **Dataset:** City of Seattle Active Business License Tax Certificate (filtered to coffee businesses)
- **Source URL:** https://data.seattle.gov/resource/wnbq-64tb.json
- **Date:** May 2026

### Practitioner use
A city planner or business analyst could use these findings to identify neighborhoods with growing or saturated coffee markets.


In [1]:
!pip install jupyter plotly kaleido pandas


In [ ]:
import pandas as pd
import plotly.express as px


---


## Section 2: Data Loading and Profile


In [2]:
# Load dataset
csv_path = "seattle_coffee_licenses.csv"
df = pd.read_csv(csv_path)
df.head()


,trade_name,city,zip,license_start_date,expiration_date,naics_description
0,KAKAO COFFEE,SEATTLE,98109,20180101,NaN,Snack and Nonalcoholic Beverage Bars
1,70 & SUNNY COFFEE CO,SEATTLE,98121-1135,20251001,NaN,Snack and Nonalcoholic Beverage Bars
2,ACQUAINTANCE COFFEE,SEATTLE,98104-2024,20231210,NaN,Mobile Food Services
3,AFRIK GROCERY & COFFEE SHOP LLC,SEATTLE,98108,20150101,NaN,Convenience Retailers
4,ALCHEMY HARVEST COFFEE,SEATTLE,98126-3325,20250201,NaN,All Other Specialty Food Retailers


In [ ]:
df.info()


In [ ]:
df.describe()


The describe() output confirms that license_start_date is the only numeric column worth summarizing. The range runs from 1981 to 2026, which tells us the dataset captures decades of licensing history, not just recent years.

In [3]:
# Quick profile checks
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nDtypes:")
print(df.dtypes)
print("\nMissing values by column:")
print(df.isna().sum())


Shape: (263, 6)
Columns: ['trade_name', 'city', 'zip', 'license_start_date', 'expiration_date', 'naics_description']

Dtypes:
trade_name             object
city                   object
zip                    object
license_start_date      int64
expiration_date       float64
naics_description      object
dtype: object

Missing values by column:
trade_name              0
city                    0
zip                     0
license_start_date      0
expiration_date       263
naics_description       0
dtype: int64


**Data profile notes:**

- The dataset has **263 rows** and **6 columns**.
- `expiration_date` is entirely blank (all missing values).
- `license_start_date` is stored as an integer in `YYYYMMDD` format (for example, `20180101`).
- `zip` contains both five-digit and ZIP+4 formats, so it needs normalization for geographic summaries.


---


## Section 3: Analytical Questions


**Question 1:** Which zip codes in Seattle have the highest concentration of licensed coffee businesses?


In [4]:
# Q1 analysis: normalize ZIP to 5 digits and compute top 10
q1 = df.copy()
q1["zip_5"] = q1["zip"].astype(str).str.extract(r"(\d{5})")
zip_counts = q1["zip_5"].value_counts(dropna=True).head(10).reset_index()
zip_counts.columns = ["zip_5", "license_count"]
zip_counts


,zip_5,license_count
0,98101,27
1,98104,21
2,98103,18
3,98109,15
4,98105,15
5,98107,15
6,98122,14
7,98121,11
8,98134,9
9,98112,8


**Interpretation:**

The top ZIP codes are concentrated in central Seattle areas, suggesting that coffee businesses cluster where foot traffic and commercial density are highest.


**Question 2:** How has the number of new coffee business licenses changed year over year?


In [5]:
# Q2 analysis: parse start year from integer date and count by year
q2 = df.copy()
q2["start_year"] = q2["license_start_date"].astype(str).str[:4]
licenses_per_year = q2["start_year"].value_counts().sort_index().reset_index()
licenses_per_year.columns = ["start_year", "new_licenses"]
licenses_per_year


,start_year,new_licenses
0,1981,1
1,1989,4
2,1990,1
3,1991,1
4,1992,1
5,1993,3
6,1994,2
7,1995,2
8,1997,3
9,1998,2


**Interpretation:**

New coffee licenses generally increase in recent years, with a notable rise near the end of the timeline, indicating accelerated market entry.


**Question 3:** Do coffee shop license starts cluster in particular zip codes during particular years?


In [6]:
# Q3 analysis: count combinations of ZIP and year, show strongest clusters
q3 = df.copy()
q3["zip_5"] = q3["zip"].astype(str).str.extract(r"(\d{5})")
q3["start_year"] = q3["license_start_date"].astype(str).str[:4]
zip_year_counts = (
    q3.groupby(["zip_5", "start_year"], dropna=True)
      .size()
      .reset_index(name="license_count")
      .sort_values("license_count", ascending=False)
)
zip_year_counts.head(15)


,zip_5,start_year,license_count
47,98103,2025,5
32,98101,2025,4
30,98101,2022,4
61,98104,2025,3
86,98107,2022,3
144,98121,2025,3
53,98104,2012,3
105,98109,2025,3
24,98101,2012,3
72,98105,2021,3


**Interpretation:**

The highest ZIP-year pairs confirm that new coffee licenses are not evenly distributed over space and time; they cluster in specific neighborhoods during specific periods.


**Question 4:** Which business category (`naics_description`) is most common among Seattle coffee shops?


In [7]:
# Q4 analysis: NAICS frequency
naics_counts = df["naics_description"].value_counts(dropna=False).reset_index()
naics_counts.columns = ["naics_description", "license_count"]
naics_counts


,naics_description,license_count
0,Snack and Nonalcoholic Beverage Bars,84
1,Limited-Service Restaurants,60
2,Mobile Food Services,25
3,Coffee and Tea Manufacturing,22
4,All Other Specialty Food Retailers,16
5,Drinking Places (Alcoholic Beverages),11
6,"Cafeterias, Grill Buffets, and Buffets",7
7,Other Grocery and Related Products Merchant Wh...,4
8,Caterers,4
9,Full-Service Restaurants,4


**Interpretation:**

`Snack and Nonalcoholic Beverage Bars` is the dominant category, which aligns with how many coffee businesses are classified in licensing records.


---


## Section 4: Visualizations


In [8]:
# Chart 1: Top 10 ZIP codes by count (bar)
fig1 = px.bar(
    zip_counts,
    x="zip_5",
    y="license_count",
    labels={"zip_5": "ZIP Code (5-digit)", "license_count": "Number of coffee licenses"},
    title="Downtown Seattle zip codes dominate coffee business licenses"
)
fig1.write_image("chart1_zip_counts.png")
fig1.show()


**Chart rationale:** A bar chart is appropriate for comparing counts across discrete ZIP categories. The plot highlights that a small number of ZIP codes account for a large share of coffee licenses.


In [9]:
# Chart 2: New licenses per year (line)
licenses_per_year_plot = licenses_per_year.copy()
licenses_per_year_plot["start_year"] = licenses_per_year_plot["start_year"].astype(int)

fig2 = px.line(
    licenses_per_year_plot,
    x="start_year",
    y="new_licenses",
    markers=True,
    labels={"start_year": "License start year", "new_licenses": "New coffee licenses"},
    title="Coffee business licensing spiked in 2025"
)
fig2.write_image("chart2_licenses_per_year.png")
fig2.show()


**Chart rationale:** A line chart is best for year-over-year trend analysis. It makes the late-period increase especially visible and supports the interpretation of a recent spike.


In [10]:
# Chart 3: ZIP vs year density heatmap
zip_year_heat = q3.groupby(["zip_5", "start_year"], dropna=True).size().reset_index(name="count")
zip_year_heat["start_year"] = zip_year_heat["start_year"].astype(int)

fig3 = px.density_heatmap(
    zip_year_heat,
    x="start_year",
    y="zip_5",
    z="count",
    labels={"start_year": "License start year", "zip_5": "ZIP Code (5-digit)", "count": "License count"},
    title="License activity clusters in central Seattle zip codes post-2020"
)
fig3.write_image("chart3_zip_year_heatmap.png")
fig3.show()


**Chart rationale:** A heatmap is ideal for spotting concentration across two dimensions (time and location). Darker cells reveal where ZIP-year combinations have the most activity.


In [11]:
# Chart 4: Top NAICS categories (horizontal bar)
naics_top8 = naics_counts.head(8).sort_values("license_count", ascending=True)

fig4 = px.bar(
    naics_top8,
    x="license_count",
    y="naics_description",
    orientation="h",
    labels={"license_count": "Number of coffee licenses", "naics_description": "NAICS description"},
    title="Most Seattle coffee shops are classified as snack bars not restaurants"
)
fig4.write_image("chart4_naics_categories.png")
fig4.show()


**Chart rationale:** A horizontal bar chart improves readability for long category names and clearly shows relative magnitude across top NAICS groups.


---


## Section 6: Conclusion

This analysis shows that Seattle coffee business licenses are concentrated in a handful of ZIP codes, with central areas appearing most saturated. Year-over-year counts indicate stronger growth in recent years, with a visible jump near 2025. The strongest ZIP-year combinations suggest localized bursts of licensing activity rather than uniform citywide expansion. One surprising point is how dominant a single NAICS category is compared with all others. A key limitation is that this is a licensing dataset snapshot; it does not include business closures, revenue performance, or neighborhood demand indicators.


## Section 5 — Process Notes

This project started in Week 4 when I built a script to pull data from the City of Seattle Open Data API. I originally explored other APIs including Yelp and API-Football, but ran into authentication barriers and signup issues. The Seattle Open Data portal worked immediately with no key required, and filtering by "COFFEE" in the trade name gave a clean, manageable dataset of 263 records.

The pivot to coffee businesses was partly practical and partly genuine curiosity. Seattle is known for its coffee culture and I wanted to see if the licensing data reflected that reputation geographically and over time.

One thing I did not expect was how many coffee businesses fall outside the "Snack and Nonalcoholic Beverage Bars" category. Mobile food services, caterers, and even consulting firms showed up in the results, which changed how I thought about Question 4.

The charts were built iteratively in Cursor using Agent Mode. The heatmap for Question 3 went through two versions before the zip-year clustering became readable. The AI suggested using density_heatmap which worked better than the grouped bar I initially tried.

Most of the markdown interpretation was written by hand after seeing the actual outputs, not before.
